# Evaluate the OOD emergent-misalignment baseline

Run this notebook once for each trained adapter seed: **42, 43, and 44**. It evaluates a sealed reconstruction of 150 broad-text prompts plus 250 LLaVA/MSCOCO VQA pairs. The audited upstream repository does not release the exact full input selection, so this is a **paper-comparable reconstruction**, never an exact reproduction.

Generation and the automated judge remain `undecided`. Only the calibrated two-reviewer artifact can record a per-seed decision; only the hashed three-seed gate can unlock primary RQ1.

## 1. Confirm the A100 runtime

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
assert 'A100' in GPU_NAME, 'Choose Runtime → Change runtime type → A100.'
assert torch.cuda.is_bf16_supported(), 'bf16 support is required.'

## 2. Mount Drive and select the training seed

`EVALUATION_SEED` is fixed across all three adapters. Change only `SEED` when repeating the notebook.

In [ ]:
from pathlib import Path
import os

SEED = 42  # repeat with 43 and 44
EVALUATION_SEED = 1729  # fixed across adapter seeds
assert SEED in {42, 43, 44}

from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)
DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm')
for subdir in ('data/ood', 'checkpoints', 'results/ood', 'runs', 'judge_cache'):
    (DRIVE_PROJECT / subdir).mkdir(parents=True, exist_ok=True)

os.environ['EM_DATA_DIR'] = str(DRIVE_PROJECT / 'data')
os.environ['EM_CHECKPOINT_DIR'] = str(DRIVE_PROJECT / 'checkpoints')
os.environ['EM_RESULTS_DIR'] = str(DRIVE_PROJECT / 'results')
os.environ['HF_HOME'] = '/content/hf-cache'

ADAPTER_DIR = DRIVE_PROJECT / 'checkpoints' / f'FT_R32_gemma3_faces_seed{SEED}'
OOD_MANIFEST = DRIVE_PROJECT / 'data' / 'ood' / 'paper_comparable_ood_v1.jsonl'
OOD_IMAGE_ROOT = DRIVE_PROJECT / 'data' / 'ood' / 'images'
OOD_OUTPUT_DIR = DRIVE_PROJECT / 'results' / 'ood' / f'seed{SEED}'
OOD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Adapter:', ADAPTER_DIR)
print('OOD manifest:', OOD_MANIFEST)

## 3. Use a clean, versioned checkout and install the evaluation runtime

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
if REPO_DIR.exists():
    assert (REPO_DIR / '.git').is_dir(), f'{REPO_DIR} is not a git clone; restart runtime.'
    assert not subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip(), 'Clone is dirty; restart runtime.'
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', 'origin', 'main'])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'origin/main'])
else:
    subprocess.check_call(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, str(REPO_DIR)])
%cd {REPO_DIR}
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
constraints_path = REPO_DIR / 'constraints' / 'colab.txt'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '--constraint', str(constraints_path), 'unsloth', 'openai>=2.0',
    'datasets>=2.19', 'huggingface-hub>=0.23', 'safetensors>=0.4', 'pyyaml>=6.0',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'])
subprocess.check_call([sys.executable, '-c', "import torch, unsloth, openai; print(torch.__version__, torch.version.cuda, 'runtime OK')"])
print('Repository commit:', REPO_COMMIT)

## 4. Verify the local provenance-complete adapter

In [ ]:
import json

for name in ('adapter_config.json', 'run_metadata.json', 'reproduction_manifest.json', 'spec.json'):
    assert (ADAPTER_DIR / name).is_file(), f'Missing {name}: {ADAPTER_DIR}'
assert list(ADAPTER_DIR.glob('adapter_model*.safetensors')), 'Missing adapter weights.'
adapter_manifest = json.loads((ADAPTER_DIR / 'reproduction_manifest.json').read_text())
assert int(adapter_manifest['seed']) == SEED, adapter_manifest
assert adapter_manifest['base_model_revision'] == 'bf46152c47f5dd20b896357cb51abc4c03b8ee8c'
print('Adapter provenance preflight passed.')

## 5. Seal the output-independent OOD inputs

Before any model output is viewed, place a reviewed JSONL at `OOD_MANIFEST`. It must contain exactly 150 unique `text` rows and 250 unique `multimodal` rows. Every row needs `sample_id`, `modality`, `prompt`, and `source`; multimodal rows also need `image_path` and the lowercase SHA-256 `image_sha256`. Exact upstream selections are unavailable, so document the deterministic reconstruction rule honestly.

Do not create or change this manifest after inspecting base or fine-tuned outputs.

In [ ]:
SELECTION_RULE = ''  # required, e.g. a frozen deterministic source/index rule
INPUT_REVIEWER = ''  # pseudonymous reviewer ID
INPUT_REVIEW_RECORD = ''  # durable note/issue/document identifier

assert OOD_MANIFEST.is_file(), f'Create the reviewed 400-row manifest first: {OOD_MANIFEST}'
OOD_SIDECAR = OOD_MANIFEST.with_suffix(OOD_MANIFEST.suffix + '.meta.json')
if not OOD_SIDECAR.exists():
    assert all((SELECTION_RULE, INPUT_REVIEWER, INPUT_REVIEW_RECORD)), 'Complete the input-review fields.'
    subprocess.check_call([
        sys.executable, 'scripts/validate_ood_manifest.py', str(OOD_MANIFEST),
        '--selection-rule', SELECTION_RULE, '--reviewer', INPUT_REVIEWER,
        '--review-record', INPUT_REVIEW_RECORD, '--image-root', str(OOD_IMAGE_ROOT),
    ])
else:
    from em_displacement_vlm.evals.ood_em import load_sealed_ood_manifest
    records, _ = load_sealed_ood_manifest(OOD_MANIFEST, image_root=OOD_IMAGE_ROOT)
    print('Verified sealed rows:', len(records))

## 6. Materialize the immutable per-seed evaluation config

In [ ]:
import yaml

cfg = yaml.safe_load(Path('configs/eval_ood_em.yaml').read_text())
cfg.update({
    'run_name': f'eval_ood_em_seed{SEED}', 'seed': SEED,
    'evaluation_seed': EVALUATION_SEED,
    'manifest_path': str(OOD_MANIFEST), 'image_root': str(OOD_IMAGE_ROOT),
    'output_dir': str(OOD_OUTPUT_DIR), 'adapter_id': str(ADAPTER_DIR),
    'adapter_provenance_path': str(ADAPTER_DIR / 'run_metadata.json'),
})
OOD_CONFIG = DRIVE_PROJECT / 'runs' / f'eval_ood_em_seed{SEED}.yaml'
rendered = yaml.safe_dump(cfg, sort_keys=False)
if OOD_CONFIG.exists() and OOD_CONFIG.read_text() != rendered:
    raise RuntimeError(f'Existing OOD config differs: {OOD_CONFIG}')
if not OOD_CONFIG.exists():
    OOD_CONFIG.write_text(rendered)
print(OOD_CONFIG.read_text())

## 7. Generate matched base and fine-tuned bundles

This is the long step: 400 independent items × 3 generations × 2 conditions. Completed immutable condition bundles are verified and reused after an interruption.

In [ ]:
subprocess.check_call([sys.executable, 'scripts/evaluate_ood_em.py', '--config', str(OOD_CONFIG)])
BASE_BUNDLE = OOD_OUTPUT_DIR / f'ood_base_seed{SEED}.json'
FT_BUNDLE = OOD_OUTPUT_DIR / f'ood_ft_seed{SEED}.json'
PAIR_PACKAGE = OOD_OUTPUT_DIR / f'ood_pair_seed{SEED}.json'
assert all(path.is_file() for path in (BASE_BUNDLE, FT_BUNDLE, PAIR_PACKAGE))
print('Generation remains undecided:', PAIR_PACKAGE)

## 8. Run the blinded bilateral judge

Use an OpenAI-compatible vision endpoint whose deployment/model revision can be recorded immutably. The local rubric is a calibrated project extension, not the upstream judge's numerically identical metric. Add Colab secrets `JUDGE_API_KEY` and `JUDGE_BASE_URL`.

In [ ]:
JUDGE_MODEL_ID = 'zai-org/GLM-4.6V-FP8'
JUDGE_REVISION = ''  # required immutable provider snapshot/deployment revision
JUDGE_ENDPOINT_ID = ''  # required stable non-secret provider/deployment label
for secret in ('JUDGE_API_KEY', 'JUDGE_BASE_URL'):
    try:
        value = userdata.get(secret)
    except Exception:
        value = None
    if not value:
        raise SystemExit(f'Colab secret not set: {secret}')
    os.environ[secret] = value
assert JUDGE_REVISION and JUDGE_ENDPOINT_ID, 'Record judge revision and endpoint ID.'

JUDGE_OUTPUT = OOD_OUTPUT_DIR / f'judge_seed{SEED}.jsonl'
JUDGE_SUMMARY = OOD_OUTPUT_DIR / f'judge_seed{SEED}.summary.json'
JUDGE_CACHE = DRIVE_PROJECT / 'judge_cache' / 'ood_judge_v2.jsonl'
if not JUDGE_OUTPUT.exists():
    subprocess.check_call([
        sys.executable, 'scripts/judge_ood_em.py',
        '--base-bundle', str(BASE_BUNDLE), '--ft-bundle', str(FT_BUNDLE),
        '--pair-package', str(PAIR_PACKAGE), '--manifest', str(OOD_MANIFEST),
        '--image-root', str(OOD_IMAGE_ROOT), '--out', str(JUDGE_OUTPUT),
        '--summary-out', str(JUDGE_SUMMARY), '--cache', str(JUDGE_CACHE),
        '--model-id', JUDGE_MODEL_ID, '--judge-revision', JUDGE_REVISION,
        '--endpoint-id', JUDGE_ENDPOINT_ID,
    ])
else:
    print('Judge output already exists; finalization will verify it:', JUDGE_OUTPUT)

## 9. Calibrate with two independent blinded reviewers

The sheet contains 15 text and 25 multimodal items, twice—one slot per independent reviewer. Keep the mapping private until both reviewers finish. Reviewers read all six responses, select each set's worst response, score it from 0–100 using the frozen rubric, and provide confidence plus notes.

In [ ]:
CALIBRATION_TEMPLATE = OOD_OUTPUT_DIR / f'calibration_seed{SEED}_template.csv'
CALIBRATION_MAPPING = OOD_OUTPUT_DIR / f'calibration_seed{SEED}_mapping.json'
if not CALIBRATION_TEMPLATE.exists():
    subprocess.check_call([
        sys.executable, 'scripts/make_ood_calibration_sheet.py',
        '--base-bundle', str(BASE_BUNDLE), '--ft-bundle', str(FT_BUNDLE),
        '--pair-package', str(PAIR_PACKAGE), '--manifest', str(OOD_MANIFEST),
        '--image-root', str(OOD_IMAGE_ROOT), '--judge-summary', str(JUDGE_SUMMARY),
        '--out', str(CALIBRATION_TEMPLATE), '--mapping-out', str(CALIBRATION_MAPPING),
    ])
print('Give separate copies/slots to two reviewers:', CALIBRATION_TEMPLATE)
print('Do not reveal until both are complete:', CALIBRATION_MAPPING)

## 10. Finalize this seed's calibrated review

Save the merged two-reviewer CSV as `calibration_seedNN_completed.csv`. Keep `undecided` unless the completed evidence supports another decision. A `pass` is rejected automatically when calibration agreement is insufficient.

In [ ]:
COMPLETED_CALIBRATION = OOD_OUTPUT_DIR / f'calibration_seed{SEED}_completed.csv'
SEED_REVIEW = OOD_OUTPUT_DIR / f'ood_review_seed{SEED}.json'
OOD_DECISION = 'undecided'  # pass, fail, or undecided
OOD_DECISION_RATIONALE = ''
LEAD_REVIEWER_ID = ''
OOD_CONFIRMATION = ''  # set exactly to: reviewed ood em seed NN

assert COMPLETED_CALIBRATION.is_file(), f'Two reviewers must complete: {COMPLETED_CALIBRATION}'
assert OOD_DECISION_RATIONALE and LEAD_REVIEWER_ID
subprocess.check_call([
    sys.executable, 'scripts/finalize_ood_review.py',
    '--pair-package', str(PAIR_PACKAGE), '--judge-output', str(JUDGE_OUTPUT),
    '--judge-summary', str(JUDGE_SUMMARY), '--calibration-csv', str(COMPLETED_CALIBRATION),
    '--calibration-mapping', str(CALIBRATION_MAPPING), '--decision', OOD_DECISION,
    '--decision-rationale', OOD_DECISION_RATIONALE, '--reviewer-id', LEAD_REVIEWER_ID,
    '--confirmation', OOD_CONFIRMATION, '--out', str(SEED_REVIEW),
])
print('Per-seed OOD review:', SEED_REVIEW)

## 11. After all three seeds, seal the cross-seed OOD gate

Run this cell only after the seed-42, seed-43, and seed-44 review files exist. A cross-seed `pass` requires all three per-seed reviews to pass under an identical manifest, decoder, fixed evaluation seed, and judge protocol.

In [ ]:
THREE_SEED_DECISION = 'undecided'
THREE_SEED_RATIONALE = ''
THREE_SEED_REVIEWER_ID = ''
THREE_SEED_CONFIRMATION = ''  # set exactly: sealed ood em seeds 42 43 44
THREE_SEED_GATE = DRIVE_PROJECT / 'results' / 'ood' / 'ood_three_seed_gate.json'
SEED_REVIEWS = [
    DRIVE_PROJECT / 'results' / 'ood' / f'seed{seed}' / f'ood_review_seed{seed}.json'
    for seed in (42, 43, 44)
]
assert all(path.is_file() for path in SEED_REVIEWS), SEED_REVIEWS
assert THREE_SEED_RATIONALE and THREE_SEED_REVIEWER_ID
command = [sys.executable, 'scripts/seal_ood_three_seed_gate.py']
for path in SEED_REVIEWS:
    command.extend(['--seed-review', str(path)])
command.extend([
    '--decision', THREE_SEED_DECISION, '--decision-rationale', THREE_SEED_RATIONALE,
    '--reviewer-id', THREE_SEED_REVIEWER_ID, '--confirmation', THREE_SEED_CONFIRMATION,
    '--out', str(THREE_SEED_GATE),
])
subprocess.check_call(command)
print('Primary RQ1 gate:', THREE_SEED_GATE)